<a href="https://colab.research.google.com/github/Moquiuti/LangChainePython/blob/main/encadeando_cadeias_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-google-genai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 13.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

api_key = userdata.get("python-ai-integrada")
os.environ["GOOGLE_API_KEY"] = api_key

print("Chave carregada com sucesso!" if api_key else "Chave não encontrada.")

Chave carregada com sucesso!


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
import json

In [7]:
modelo = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

In [8]:
class Destino(BaseModel):
    cidade: str = Field(description="Nome da cidade recomendada")
    motivo: str = Field(description="Motivo pelo qual a cidade foi recomendada")

In [9]:
parser_destino = JsonOutputParser(pydantic_object=Destino)

In [11]:
from langchain_core.prompts import PromptTemplate

prompt_cidade = PromptTemplate(
    template="""
Sugira uma cidade com base no interesse informado.

Interesse: {interesse}

Responda no formato especificado abaixo:
{format_instructions}
""",
    input_variables=["interesse"],
    partial_variables={"format_instructions": parser_destino.get_format_instructions()}
)

In [12]:
cadeia_cidade = prompt_cidade | modelo | parser_destino

In [13]:
resposta_cidade = cadeia_cidade.invoke({"interesse": "praias"})
print(resposta_cidade)

{'cidade': 'Rio de Janeiro', 'motivo': 'A cidade é mundialmente famosa por suas praias icônicas como Copacabana e Ipanema, que oferecem beleza natural, calçadões vibrantes, diversas atividades aquáticas e uma atmosfera única, perfeita para quem busca o sol e o mar.'}


In [14]:
class Restaurantes(BaseModel):
    restaurantes: list[str] = Field(description="Lista de restaurantes recomendados na cidade")

In [15]:
parser_restaurantes = JsonOutputParser(pydantic_object=Restaurantes)

In [16]:
prompt_restaurantes = PromptTemplate(
    template="""
Com base na cidade abaixo, sugira 3 restaurantes interessantes para visitar.

Cidade: {cidade}

Responda no formato especificado abaixo:
{format_instructions}
""",
    input_variables=["cidade"],
    partial_variables={"format_instructions": parser_restaurantes.get_format_instructions()}
)

In [17]:
cadeia_restaurantes = prompt_restaurantes | modelo | parser_restaurantes

In [18]:
resposta_restaurantes = cadeia_restaurantes.invoke({"cidade": resposta_cidade["cidade"]})
print(resposta_restaurantes)

{'restaurantes': ['Oro', 'Aprazível', 'Mee Restaurante']}


In [19]:
prompt_cultural = PromptTemplate(
    template="""
Monte uma sugestão final de passeio em tom amigável e organizado, usando as informações abaixo.

Cidade recomendada: {cidade}
Motivo da escolha: {motivo}
Restaurantes sugeridos: {restaurantes}

Inclua também uma dica cultural ou de lazer para complementar a viagem.
""",
    input_variables=["cidade", "motivo", "restaurantes"]
)

In [20]:
cadeia_cultural = prompt_cultural | modelo | StrOutputParser()

In [21]:
resposta_final = cadeia_cultural.invoke({
    "cidade": resposta_cidade["cidade"],
    "motivo": resposta_cidade["motivo"],
    "restaurantes": ", ".join(resposta_restaurantes["restaurantes"])
})

print(resposta_final)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Fluxo completo com debug

In [22]:
def executar_fluxo(interesse: str):
    print("=== ETAPA 1: BUSCANDO DESTINO ===")
    destino = cadeia_cidade.invoke({"interesse": interesse})
    print(destino)
    print()

    print("=== ETAPA 2: BUSCANDO RESTAURANTES ===")
    restaurantes = cadeia_restaurantes.invoke({"cidade": destino["cidade"]})
    print(restaurantes)
    print()

    print("=== ETAPA 3: GERANDO RESPOSTA FINAL ===")
    resposta = cadeia_cultural.invoke({
        "cidade": destino["cidade"],
        "motivo": destino["motivo"],
        "restaurantes": ", ".join(restaurantes["restaurantes"])
    })
    print(resposta)
    print()

    return {
        "destino": destino,
        "restaurantes": restaurantes,
        "resposta_final": resposta
    }

In [23]:
resultado = executar_fluxo("praias")

=== ETAPA 1: BUSCANDO DESTINO ===
{'cidade': 'Rio de Janeiro', 'motivo': 'Famosa mundialmente por suas praias icônicas como Copacabana e Ipanema, além de oferecer uma beleza natural deslumbrante e uma vibrante cultura praiana.'}

=== ETAPA 2: BUSCANDO RESTAURANTES ===
{'restaurantes': ['Mee', 'Oteque', 'Aprazível']}

=== ETAPA 3: GERANDO RESPOSTA FINAL ===


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 13.234828533s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT